In [ ]:
# ECNN ablation on processed BraTS test set (score and threshold methods)
import os
import sys
import zipfile
import subprocess
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve,
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
 )

# =========================
# ENV / PATHS
# =========================
try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
except Exception:
    pass

if Path('/content').exists():
    REPO_ROOT = Path('/content/symAD-ECNN')
    if not REPO_ROOT.exists():
        subprocess.check_call(['git', 'clone', 'https://github.com/RifaDeen/symAD-ECNN.git', str(REPO_ROOT)])
    PROJECT_ROOT = Path('/content/drive/MyDrive/symAD-ECNN') if Path('/content/drive/MyDrive').exists() else REPO_ROOT
else:
    REPO_ROOT = Path.cwd()
    PROJECT_ROOT = Path.cwd()

ECNN_HELPER_DIR = REPO_ROOT / 'notebooks' / 'evals' / 'ecnn_thresholding'
for p in [REPO_ROOT, ECNN_HELPER_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

def ensure_e2cnn_installed():
    if importlib.util.find_spec('e2cnn') is not None:
        return
    print('e2cnn not found. Installing...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'e2cnn'])
    if importlib.util.find_spec('e2cnn') is None:
        raise RuntimeError('Failed to install e2cnn. Please run: pip install e2cnn')
    print('e2cnn installed successfully.')

ensure_e2cnn_installed()
from ecnn_model_loader import get_model_for_inference

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

# Processed data locations (prefer extracted folder, fallback to zip extraction)
DATA_DIR = Path('/content/test_eval_data/processed') if Path('/content').exists() else PROJECT_ROOT / 'data' / 'test_eval_data' / 'processed'
PROCESSED_ZIP = PROJECT_ROOT / 'data' / 'brats_test_with_masks_processed.zip'
IMAGE_DIR = DATA_DIR / 'images'
LABEL_DIR = DATA_DIR / 'labels'

if (not IMAGE_DIR.exists() or not LABEL_DIR.exists()) and PROCESSED_ZIP.exists():
    print(f'Extracting processed zip: {PROCESSED_ZIP}')
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(PROCESSED_ZIP, 'r') as zf:
        zf.extractall(DATA_DIR.parent)

if not IMAGE_DIR.exists() or not LABEL_DIR.exists():
    raise FileNotFoundError(f'Processed data not found at {DATA_DIR}. Run test_preprocessing first.')

MODELS_ROOT = PROJECT_ROOT / 'models' / 'saved_models'
OUT_DIR = PROJECT_ROOT / 'evaluations' / 'tables'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / 'ecnn_ablation_results.csv'

def find_ecnn_checkpoint() -> Path:
    patterns = ['ecnn_optimized_best.pth', 'ecnn_best.pth', '*ecnn*best*.pth']
    search_roots = [MODELS_ROOT / 'ecnn_optimized', MODELS_ROOT]
    for root in search_roots:
        if not root.exists():
            continue
        for pat in patterns:
            hits = sorted(root.rglob(pat))
            if hits:
                return hits[0]
    raise FileNotFoundError(f'ECNN checkpoint not found under {MODELS_ROOT}')

MODEL_PATH = find_ecnn_checkpoint()
print(f'ECNN checkpoint: {MODEL_PATH}')

# =========================
# LOAD MODEL
# =========================
model, _ = get_model_for_inference(str(MODEL_PATH), str(DEVICE))
model = model.to(DEVICE).eval()

# =========================
# DATA LOADER (simple loop)
# =========================
image_files = sorted(IMAGE_DIR.glob('slice_*.npy'))
if len(image_files) == 0:
    raise RuntimeError(f'No images found in {IMAGE_DIR}')

def load_sample(img_fp: Path):
    sid = img_fp.stem.split('_')[-1]
    label_fp = LABEL_DIR / f'label_{sid}.npy'
    if not label_fp.exists():
        raise FileNotFoundError(f'Missing label for {img_fp.name}: {label_fp}')
    img = np.load(img_fp).astype(np.float32)
    lab = int(np.load(label_fp)[0])
    return img, lab

# =========================
# SCORE EXTRACTION
# =========================
all_errors = []
all_labels = []

with torch.no_grad():
    for img_fp in tqdm(image_files, desc='Running ECNN inference'):
        img, label = load_sample(img_fp)
        x = torch.from_numpy(img).unsqueeze(0).unsqueeze(0).to(DEVICE)
        recon = model(x)
        recon_np = recon.detach().cpu().numpy()[0, 0]

        mae_map = np.abs(img - recon_np).astype(np.float32)
        mse_map = ((img - recon_np) ** 2).astype(np.float32)
        all_errors.append({'mae': mae_map, 'mse': mse_map})
        all_labels.append(label)

y_true = np.asarray(all_labels, dtype=np.int32)
print(f'Samples: {len(y_true)} | Normal={(y_true==0).sum()} | Anomaly={(y_true==1).sum()}')

def get_score(err, method: str) -> float:
    if method == 'mean_mse':
        return float(err['mse'].mean())
    if method == 'mae':
        return float(err['mae'].mean())
    if method == 'p95':
        return float(np.percentile(err['mae'], 95))
    raise ValueError(method)

def threshold_youden(y, scores):
    # Note: label-informed (for ablation only)
    fpr, tpr, thr = roc_curve(y, scores)
    return float(thr[np.argmax(tpr - fpr)])

def threshold_fpr(normal_scores, target_fpr=0.2):
    return float(np.percentile(normal_scores, 100 * (1 - target_fpr)))

score_methods = ['mean_mse', 'mae', 'p95']
threshold_methods = ['fpr_10', 'fpr_20', 'youden']

rows = []
for sm in score_methods:
    scores = np.asarray([get_score(e, sm) for e in all_errors], dtype=np.float32)
    normal_scores = scores[y_true == 0]
    if len(normal_scores) == 0:
        raise RuntimeError('No normal samples found for thresholding.')

    for tm in threshold_methods:
        if tm == 'fpr_10':
            thr = threshold_fpr(normal_scores, 0.10)
            leakage_risk = False
        elif tm == 'fpr_20':
            thr = threshold_fpr(normal_scores, 0.20)
            leakage_risk = False
        else:
            thr = threshold_youden(y_true, scores)
            leakage_risk = True

        y_pred = (scores >= thr).astype(np.int32)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
        auprc = average_precision_score(y_true, scores)

        rows.append({
            'model': 'ECNN-AE (Optimized)',
            'score_method': sm,
            'threshold_method': tm,
            'threshold': thr,
            'leakage_risk': leakage_risk,
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
            'fpr': fp / (fp + tn) if (fp + tn) > 0 else 0.0,
            'fnr': fn / (fn + tp) if (fn + tp) > 0 else 0.0,
            'auroc': roc_auc_score(y_true, scores) if len(np.unique(y_true)) > 1 else 0.5,
            'auprc': auprc,
            'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
            'n': int(len(y_true)),
            'checkpoint': str(MODEL_PATH),
        })

df = pd.DataFrame(rows).sort_values(['f1', 'auroc'], ascending=False).reset_index(drop=True)
display(df)

print('\n=== FPR ≤ 10% ONLY ===')
display(df[df['fpr'] <= 0.10])

df.to_csv(OUT_CSV, index=False)
print(f'\nSaved: {OUT_CSV}')